In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install -U langchain langchain-community langchain-chroma chromadb

In [3]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Creating Document

In [4]:
from langchain_core.documents import Document

In [5]:
doc1 = Document(
    page_content="Virat Kohli is a top-order batsman for India known as the Run Machine. He is the ultimate Chase Master who has won many games for his team while batting second.",
    metadata={"team": "India"}
)

doc2 = Document(
    page_content="Rohit Sharma is an opening batsman for India and is famously called the Hitman. He is a powerful batter known for hitting big sixes and leading the team as captain.",
    metadata={"team": "India"}
)

doc3 = Document(
    page_content="Mitchell Starc is a left-arm fast bowler for Australia. He is a specialist bowler known for his high speed and yorkers that can bowl out any batsman in the world.",
    metadata={"team": "Australia"}
)

doc4 = Document(
    page_content="Ben Stokes is a star all-rounder for England. He is a world-class player who contributes as both a powerful batter and a reliable bowler in high-pressure matches.",
    metadata={"team": "England"}
)

doc5 = Document(
    page_content="AB de Villiers is a legendary batsman for South Africa known as Mr. 360. He is a versatile batter who can hit shots in every direction of the cricket ground.",
    metadata={"team": "South Africa"}
)

In [6]:
docs = [doc1, doc2, doc3, doc4, doc5]

# Create Vector Store

In [7]:
from langchain_chroma import Chroma

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

#### create vector store

In [9]:
vector_store = Chroma(
    embedding_function = embeddings,
    persist_directory='my_chroma_db',
    collection_name='cricket_players'
)

In [10]:
#adding documents
vector_store.add_documents(docs)  #returns ids for each document

['cc2573f1-4eab-4aea-99c6-1d7ab1b2e8b9',
 '916ee8c5-9820-4f18-9a0e-e973e7b6f426',
 '0bc4c542-bf7e-4953-9b9c-5b372b4eb948',
 '8fedb6a6-86b5-491e-af59-1263f9d51969',
 '5dcf5473-226c-43a2-9463-bd7b79240229']

# Viewing documents

In [11]:
vector_store.get(
    include=['embeddings','documents','metadatas']
)

{'ids': ['cc2573f1-4eab-4aea-99c6-1d7ab1b2e8b9',
  '916ee8c5-9820-4f18-9a0e-e973e7b6f426',
  '0bc4c542-bf7e-4953-9b9c-5b372b4eb948',
  '8fedb6a6-86b5-491e-af59-1263f9d51969',
  '5dcf5473-226c-43a2-9463-bd7b79240229'],
 'embeddings': array([[ 0.02033388,  0.06444178, -0.04853628, ..., -0.04291681,
          0.07105959,  0.01462915],
        [ 0.03876261,  0.01989105, -0.07403702, ..., -0.02351438,
          0.051408  ,  0.05230317],
        [-0.01855271, -0.03338531, -0.07047473, ..., -0.02745939,
         -0.01610561,  0.02575224],
        [ 0.00712803,  0.01640912, -0.03833336, ..., -0.00999849,
          0.04243052,  0.05601252],
        [-0.01591147,  0.04089143, -0.07736557, ..., -0.07804433,
          0.00332702, -0.01639182]]),
 'documents': ['Virat Kohli is a top-order batsman for India known as the Run Machine. He is the ultimate Chase Master who has won many games for his team while batting second.',
  'Rohit Sharma is an opening batsman for India and is famously called the Hi

# Searching Documents

In [12]:
vector_store.similarity_search(
    query='List out the bowlers.',
    k=2  #top 2 matches
)

[Document(id='0bc4c542-bf7e-4953-9b9c-5b372b4eb948', metadata={'team': 'Australia'}, page_content='Mitchell Starc is a left-arm fast bowler for Australia. He is a specialist bowler known for his high speed and yorkers that can bowl out any batsman in the world.'),
 Document(id='916ee8c5-9820-4f18-9a0e-e973e7b6f426', metadata={'team': 'India'}, page_content='Rohit Sharma is an opening batsman for India and is famously called the Hitman. He is a powerful batter known for hitting big sixes and leading the team as captain.')]

In [13]:
#with similarity scores
vector_store.similarity_search_with_score(
    query='Who among these are bowlers',
    k=2
)

[(Document(id='0bc4c542-bf7e-4953-9b9c-5b372b4eb948', metadata={'team': 'Australia'}, page_content='Mitchell Starc is a left-arm fast bowler for Australia. He is a specialist bowler known for his high speed and yorkers that can bowl out any batsman in the world.'),
  1.0606375932693481),
 (Document(id='916ee8c5-9820-4f18-9a0e-e973e7b6f426', metadata={'team': 'India'}, page_content='Rohit Sharma is an opening batsman for India and is famously called the Hitman. He is a powerful batter known for hitting big sixes and leading the team as captain.'),
  1.2248455286026)]

In [14]:
#filtering using metadata
vector_store.similarity_search_with_score(
    query="",
    filter={"team": "India"}
)

[(Document(id='cc2573f1-4eab-4aea-99c6-1d7ab1b2e8b9', metadata={'team': 'India'}, page_content='Virat Kohli is a top-order batsman for India known as the Run Machine. He is the ultimate Chase Master who has won many games for his team while batting second.'),
  1.8718632459640503),
 (Document(id='916ee8c5-9820-4f18-9a0e-e973e7b6f426', metadata={'team': 'India'}, page_content='Rohit Sharma is an opening batsman for India and is famously called the Hitman. He is a powerful batter known for hitting big sixes and leading the team as captain.'),
  1.9036788940429688)]

# Updating Documents

In [15]:
updated_doc1 = Document(
    page_content='Virat Kohli is known as the run-machine-kohli. he is also called the chase master and has led Inida to incredible victories',
    metadata={'team':'India'}
)

vector_store.update_document(document_id='6e971127-8fc0-4a45-9e62-e5060b7f5a8d',document=updated_doc1)

In [16]:
vector_store.get(include=['embeddings','documents','metadatas'])

{'ids': ['cc2573f1-4eab-4aea-99c6-1d7ab1b2e8b9',
  '916ee8c5-9820-4f18-9a0e-e973e7b6f426',
  '0bc4c542-bf7e-4953-9b9c-5b372b4eb948',
  '8fedb6a6-86b5-491e-af59-1263f9d51969',
  '5dcf5473-226c-43a2-9463-bd7b79240229'],
 'embeddings': array([[ 0.02033388,  0.06444178, -0.04853628, ..., -0.04291681,
          0.07105959,  0.01462915],
        [ 0.03876261,  0.01989105, -0.07403702, ..., -0.02351438,
          0.051408  ,  0.05230317],
        [-0.01855271, -0.03338531, -0.07047473, ..., -0.02745939,
         -0.01610561,  0.02575224],
        [ 0.00712803,  0.01640912, -0.03833336, ..., -0.00999849,
          0.04243052,  0.05601252],
        [-0.01591147,  0.04089143, -0.07736557, ..., -0.07804433,
          0.00332702, -0.01639182]]),
 'documents': ['Virat Kohli is a top-order batsman for India known as the Run Machine. He is the ultimate Chase Master who has won many games for his team while batting second.',
  'Rohit Sharma is an opening batsman for India and is famously called the Hi

# Deleting Documents

In [19]:
vector_store.delete(ids=['8fedb6a6-86b5-491e-af59-1263f9d51969'])

In [20]:
vector_store.get(include=['embeddings','documents','metadatas'])

{'ids': ['cc2573f1-4eab-4aea-99c6-1d7ab1b2e8b9',
  '916ee8c5-9820-4f18-9a0e-e973e7b6f426',
  '0bc4c542-bf7e-4953-9b9c-5b372b4eb948',
  '5dcf5473-226c-43a2-9463-bd7b79240229'],
 'embeddings': array([[ 0.02033388,  0.06444178, -0.04853628, ..., -0.04291681,
          0.07105959,  0.01462915],
        [ 0.03876261,  0.01989105, -0.07403702, ..., -0.02351438,
          0.051408  ,  0.05230317],
        [-0.01855271, -0.03338531, -0.07047473, ..., -0.02745939,
         -0.01610561,  0.02575224],
        [-0.01591147,  0.04089143, -0.07736557, ..., -0.07804433,
          0.00332702, -0.01639182]]),
 'documents': ['Virat Kohli is a top-order batsman for India known as the Run Machine. He is the ultimate Chase Master who has won many games for his team while batting second.',
  'Rohit Sharma is an opening batsman for India and is famously called the Hitman. He is a powerful batter known for hitting big sixes and leading the team as captain.',
  'Mitchell Starc is a left-arm fast bowler for Aus